# RunPod Color LoRA Notebook

This notebook runs the amphora color LoRA flow on RunPod using the current Python environment.

What it does:
- verifies repo, Python, and GPU
- installs training dependencies without creating a venv
- ensures the Diffusers example script is present
- regenerates `metadata.jsonl` for the color dataset if needed
- validates the LoRA command with a dry-run
- starts color LoRA training


In [ ]:
REPO_DIR = "/workspace/luminance-based-diffusion"
COLOR_DATA_DIR = f"{REPO_DIR}/data/vaze_bw/color/train"
COLOR_CONFIG = f"{REPO_DIR}/configs/train_lora_vaze_color.yaml"
COLOR_SCRIPT = f"{REPO_DIR}/scripts/run_train_lora_vaze_color.sh"
LOG_PATH = f"{REPO_DIR}/runs/train_lora_amphora_color/train.log"

print("REPO_DIR=", REPO_DIR)
print("COLOR_DATA_DIR=", COLOR_DATA_DIR)
print("COLOR_CONFIG=", COLOR_CONFIG)
print("COLOR_SCRIPT=", COLOR_SCRIPT)
print("LOG_PATH=", LOG_PATH)


In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion

pwd
git status --short || true
python --version
nvidia-smi


In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion

python -m pip install --upgrade pip setuptools wheel
python -m pip install -e ".[dev,train]"
python -m pip install "transformers>=4.41,<5" "peft>=0.11,<0.12"

bash scripts/setup_diffusers_examples.sh

python -c 'import torch; print("torch", torch.__version__); print("cuda", torch.cuda.is_available()); print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")'
python -c 'import accelerate, transformers, peft; print("accelerate ok"); print("transformers ok"); print("peft ok")'

test -f configs/train_lora_vaze_color.yaml
test -f scripts/run_train_lora_vaze_color.sh
test -f external/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py


## Dataset Metadata Repair

Your previous run failed because `data/vaze_bw/color/train/metadata.jsonl` was missing, so the dataset only exposed an `image` column.

Run the next cell even if you think metadata already exists. It rewrites the color and grayscale `metadata.jsonl` files from `data/vaze_bw/meta/index.csv`.

In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion

bash scripts/write_dataset_metadata.sh /workspace/luminance-based-diffusion/data/vaze_bw

echo
echo "Color train dir preview:"
python - <<'PY'
from pathlib import Path

train_dir = Path('data/vaze_bw/color/train')
for path in sorted(train_dir.iterdir())[:20]:
    stat = path.stat()
    print(f"{path.name}\t{stat.st_size} bytes")
PY

echo
echo "metadata.jsonl preview:"
python - <<'PY'
from pathlib import Path

metadata = Path('data/vaze_bw/color/train/metadata.jsonl')
with metadata.open('r', encoding='utf-8') as handle:
    for idx, line in enumerate(handle):
        if idx >= 5:
            break
        print(line.rstrip())
PY


In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion

PYTHONPATH=src python -m lbd.cli train lora --config configs/train_lora_vaze_color.yaml --dry-run


## Start Training

Run the next cell to launch color LoRA training in the notebook output stream.

In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion
PYTHONPATH=src bash scripts/run_train_lora_vaze_color.sh


## Optional Detached Training

If you want training to survive a notebook disconnect, use the next two cells instead of the direct training cell above.

In [ ]:
%%bash
set -euo pipefail

cd /workspace/luminance-based-diffusion
tmux new-session -d -s lora_color 'cd /workspace/luminance-based-diffusion && PYTHONPATH=src bash scripts/run_train_lora_vaze_color.sh'
tmux capture-pane -pt lora_color


In [ ]:
!tail -n 50 /workspace/luminance-based-diffusion/runs/train_lora_amphora_color/train.log || true
